In [1]:
# ============================================================
# MODULE 4: Hurricane Milton, from storm track to loss (+ VLM)
# Block 4.1: install libraries, set the study area
# ============================================================

# --- Install (about 1 minute in Colab) ---
# geopandas    : tables with a geometry column (points, lines, polygons)
# osmnx        : downloads OpenStreetMap data (building footprints)
# leafmap      : interactive maps inside the notebook
# google-genai : Google's Gemini API, our LLM + VLM (free tier available)
# gradio       : turns Python functions into a small web app with a share link
# (scikit-learn for the tuning step is already installed in Colab)
!pip -q install geopandas osmnx leafmap google-genai gradio

import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
import leafmap.foliumap as leafmap

# --- Study area: Siesta Key + Sarasota, Florida ---
# Hurricane Milton made landfall near Siesta Key on 9 Oct 2024
# (~8:30 pm local time) as a Category 3 hurricane.
# bbox order = (west, south, east, north) in degrees (EPSG:4326)
bbox   = (-82.57, 27.24, -82.52, 27.29)
center = (27.265, -82.545)          # (lat, lon) used to centre maps

# Metric CRS so distances come out in metres.
# UTM zone 17N covers western Florida.
METRIC_CRS = "EPSG:32617"

# --- Fixed storm setting (not a knob) ---
# Radius of maximum wind = distance from the eye where winds are strongest.
# The NHC best track gives position + max wind, but not this radius,
# so we fix a typical value for a major hurricane.
RMW_KM = 30

print("Setup done. Study area:", bbox)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 669.2/669.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.6/108.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 4.0 MB/s eta 0:00:00
Setup done. Study area: (-82.57, 27.24, -82.52, 27.29)


In [2]:
# ============================================================
# Block 4.2: Hurricane Milton's track AND size, from NOAA IBTrACS
# ============================================================
# DATA SOURCE: IBTrACS (NOAA NCEI), global best-track archive.
# Every 3 hours it records:
#   LAT, LON  = eye position          USA_WIND = max sustained wind (knots)
#   USA_PRES  = central pressure (mb) USA_RMW  = radius of max wind (nmi)
#   USA_R34_NE ... = how far 34-knot winds reached in each quadrant
#                    (NE, SE, SW, NW). Same for 50 kt and 64 kt.
#   34 kt = tropical-storm force, 50 kt = strong, 64 kt = hurricane force

url = ("https://www.ncei.noaa.gov/data/international-best-track-archive-"
       "for-climate-stewardship-ibtracs/v04r01/access/csv/"
       "ibtracs.last3years.list.v04r01.csv")

quads = ["NE", "SE", "SW", "NW"]
radii_cols = [f"USA_R{k}_{q}" for k in (34, 50, 64) for q in quads]
cols = ["SEASON", "NAME", "ISO_TIME", "LAT", "LON",
        "USA_WIND", "USA_PRES", "USA_RMW"] + radii_cols

# skiprows=[1]: the second line of the file holds units, not data
df = pd.read_csv(url, usecols=cols, skiprows=[1], low_memory=False)
milton = df[(df["NAME"] == "MILTON") & (df["SEASON"].astype(str) == "2024")].copy()

# Text -> numbers (blank cells become NaN)
num_cols = ["LAT", "LON", "USA_WIND", "USA_PRES", "USA_RMW"] + radii_cols
milton[num_cols] = milton[num_cols].apply(pd.to_numeric, errors="coerce")

# Unit conversions: knots -> mph, nautical miles -> km
milton["wind_mph"] = (milton["USA_WIND"] * 1.151).round()
milton["rmw_km"]   = (milton["USA_RMW"] * 1.852).round()

# One radius per wind level = average of the 4 quadrants (in km).
# (Real storms are lopsided; the average keeps it simple.)
for k in (34, 50, 64):
    milton[f"r{k}_km"] = milton[[f"USA_R{k}_{q}" for q in quads]].mean(axis=1) * 1.852

# Table -> GeoDataFrame of eye points
track = gpd.GeoDataFrame(
    milton, geometry=gpd.points_from_xy(milton["LON"], milton["LAT"]),
    crs="EPSG:4326")

print(len(track), "track points")
track[["ISO_TIME", "wind_mph", "rmw_km", "r34_km", "r64_km"]].tail(10)

62 track points


,ISO_TIME,wind_mph,rmw_km,r34_km,r64_km
11018,2024-10-10 15:00:00,73.0,93.0,365.770,NaN
11019,2024-10-10 18:00:00,69.0,93.0,375.030,NaN
11020,2024-10-10 21:00:00,69.0,93.0,372.715,NaN
11021,2024-10-11 00:00:00,69.0,93.0,370.400,NaN
11022,2024-10-11 03:00:00,67.0,93.0,361.140,NaN
11023,2024-10-11 06:00:00,63.0,93.0,351.880,NaN
11024,2024-10-11 09:00:00,63.0,93.0,351.880,NaN
11025,2024-10-11 12:00:00,63.0,93.0,351.880,NaN
11026,2024-10-11 15:00:00,61.0,93.0,351.880,NaN
11027,2024-10-11 18:00:00,58.0,93.0,351.880,NaN


In [3]:
# ============================================================
# Block 4.2b: draw track + wind footprint (34 / 50 / 64 kt)
# ============================================================
# Idea: draw a circle around each eye position with that time's wind
# radius, then join neighbouring circles into one band. That band is the
# "swath": everywhere that felt at least that wind speed.
import folium
from shapely.geometry import box, LineString
from shapely.ops import unary_union

DIST_CRS = "EPSG:5070"   # metric CRS for the US + Gulf (buffers in metres)

def base_map(zoom=12):
    m = folium.Map(location=center, zoom_start=zoom, tiles="CartoDB positron")
    folium.TileLayer("Esri.WorldImagery", name="Satellite").add_to(m)
    return m

def wind_swath(k):
    """Footprint of winds >= k knots along the track."""
    t = track.dropna(subset=[f"r{k}_km"])
    t = t[t[f"r{k}_km"] > 0].to_crs(DIST_CRS)
    circles = t.geometry.buffer(t[f"r{k}_km"] * 1000)     # km -> m
    # join each circle to the next one (convex hull) so the band has no gaps
    pieces = [unary_union(circles.iloc[i:i+2]).convex_hull
              for i in range(len(circles) - 1)]
    return gpd.GeoDataFrame({"wind": [f"{k} kt+"]},
                            geometry=[unary_union(pieces)],
                            crs=DIST_CRS).to_crs("EPSG:4326")

swaths = {k: wind_swath(k) for k in (34, 50, 64)}
track_line = gpd.GeoDataFrame(geometry=[LineString(track.geometry)], crs="EPSG:4326")
aoi = gpd.GeoDataFrame(geometry=[box(*bbox)], crs="EPSG:4326")

m = base_map(zoom=6)
for k, colour in [(34, "yellow"), (50, "orange"), (64, "red")]:
    swaths[k].explore(m=m, color=colour, style_kwds={"fillOpacity": 0.25},
                      name=f"{k}-knot wind footprint")
track_line.explore(m=m, color="black", name="Track")
track[["ISO_TIME", "wind_mph", "rmw_km", "geometry"]].explore(
    m=m, column="wind_mph", cmap="YlOrRd", marker_kwds={"radius": 5},
    name="Eye positions", legend=False)
aoi.explore(m=m, color="blue", name="Study area")
folium.LayerControl().add_to(m)
m

In [4]:
# ============================================================
# Block 4.3: building footprints + places (POIs) from Overture Maps
# ============================================================
# DATA SOURCE: Overture Maps Foundation, open map data from Microsoft, Meta,
# Amazon, TomTom and others. It COMBINES OpenStreetMap with buildings that
# AI models detected in satellite imagery (Microsoft, Google), so it finds
# more buildings than OSM alone.
#   buildings: footprint + height (metres) + class (house, school ...) + name
#   places   : points of interest from business listings: name + category
#              (hotel, restaurant, school, beach resort ...)
# We use Overture's official downloader, clipped to our bbox. GeoParquet is
# a fast file format for map tables.
!pip -q install overturemaps
w, s, e, n = bbox
!overturemaps download --bbox={w},{s},{e},{n} -f geoparquet --type=building -o ov_buildings.parquet
!overturemaps download --bbox={w},{s},{e},{n} -f geoparquet --type=place    -o ov_places.parquet

# Overture stores names as {"primary": "..."}; this pulls out the main name
def main_name(d):
    return d.get("primary") if isinstance(d, dict) else None

# --- Buildings ---
ovb = gpd.read_parquet("ov_buildings.parquet").to_crs("EPSG:4326")
buildings = gpd.GeoDataFrame({
    "bldg_id":  ovb["id"],
    "name":     ovb["names"].apply(main_name),
    "class":    ovb["class"],
    "height_m": ovb["height"].round(1),
}, geometry=ovb.geometry, crs="EPSG:4326")
buildings["area_m2"] = buildings.to_crs(METRIC_CRS).area.round()

# --- Places (POIs) ---
ovp = gpd.read_parquet("ov_places.parquet").to_crs("EPSG:4326")
if "basic_category" in ovp.columns:                    # newer Overture releases
    cat = ovp["basic_category"]
else:                                                  # older releases
    cat = ovp["categories"].apply(lambda c: c.get("primary") if isinstance(c, dict) else None)
places = gpd.GeoDataFrame({"place_name": ovp["names"].apply(main_name), "place_cat": cat},
                          geometry=ovp.geometry, crs="EPSG:4326")

print(len(buildings), "buildings |", len(places), "places")
print(f"Height known for {buildings['height_m'].notna().mean():.0%} of buildings; "
      f"class known for {buildings['class'].notna().mean():.0%}")
print("\nTop place categories:\n", places["place_cat"].value_counts().head(10))



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.0 MB/s eta 0:00:00
Downloading: 7752 rows [00:01, 6424.25 rows/s]
State saved to ov_buildings.parquet.state
Downloading: 1616 rows [00:00, 2500.09 rows/s]
State saved to ov_places.parquet.state
7752 buildings | 1616 places
Height known for 66% of buildings; class known for 14%

Top place categories:
 place_cat
real_estate_service               251
private_lodging                   116
restaurant                        107
personal_or_beauty_service         62
beach                              61
home_service                       59
casual_eatery                      42
hardware_home_and_garden_store     41
fashion_and_apparel_store          39
financial_service                  39
Name: count, dtype: int64


In [5]:
# ============================================================
# Block 4.5: how exposed is each building? (water, ground height, roads)
# ============================================================
# Storm surge (sea water pushed onshore) hits buildings CLOSE to the shore and
# on LOW ground hardest. Roads matter for emergency access after the storm.
#   dist_water_m : distance to shoreline. OSM "coastline" follows Gulf + bay shores
#   elev_m       : ground height above sea level. Copernicus DEM (30 m), Planetary Computer
#   dist_road_m  : distance to the nearest road. OSM roads ("highway" tag)
#   nearest_road : type of that road (primary, secondary, residential, service ...)
!pip -q install pystac-client planetary-computer rioxarray
import pystac_client, planetary_computer, rioxarray
import xarray as xr

w, s, e, n = bbox
ctr_m = buildings.to_crs(METRIC_CRS).centroid          # building centres, in metres

# 1) Distance to the shoreline (search a slightly bigger box so the shore is included)
coast = ox.features_from_bbox(bbox=(w - 0.05, s - 0.05, e + 0.05, n + 0.05),
                              tags={"natural": "coastline"})
shore = coast.to_crs(METRIC_CRS).geometry.union_all()
buildings["dist_water_m"] = ctr_m.distance(shore).round()

# 2) Ground height at each building centre
cat = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1",
                                modifier=planetary_computer.sign_inplace)
dem_item = next(cat.search(collections=["cop-dem-glo-30"], bbox=bbox).items())
dem = rioxarray.open_rasterio(dem_item.assets["data"].href).squeeze()
ctr_ll = ctr_m.to_crs("EPSG:4326")
buildings["elev_m"] = dem.sel(x=xr.DataArray(ctr_ll.x.values), y=xr.DataArray(ctr_ll.y.values),
                              method="nearest").values.round(1)

# 3) Nearest road and how far it is
roads = ox.features_from_bbox(bbox=bbox, tags={"highway": ["trunk", "primary", "secondary",
                              "tertiary", "residential", "unclassified", "service"]})
roads = roads[roads.geometry.geom_type.isin(["LineString", "MultiLineString"])].reset_index()
near = gpd.sjoin_nearest(buildings[["bldg_id", "geometry"]].to_crs(METRIC_CRS),
                         roads[["highway", "geometry"]].to_crs(METRIC_CRS),
                         distance_col="dist_road_m").drop_duplicates("bldg_id")
buildings = buildings.drop(columns=["dist_road_m", "nearest_road"], errors="ignore").merge(
    near[["bldg_id", "dist_road_m", "highway"]].rename(columns={"highway": "nearest_road"}),
    on="bldg_id", how="left")
buildings["dist_road_m"] = buildings["dist_road_m"].round()

print(buildings[["dist_water_m", "elev_m", "dist_road_m"]].describe().round(1))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 19.5 MB/s eta 0:00:00
       dist_water_m  elev_m  dist_road_m
count        7752.0  7752.0       7752.0
mean          291.4     5.4         15.5
std           338.3     2.3         11.3
min             0.0    -3.6          0.0
25%            52.0     3.7          9.0
50%           147.5     5.1         15.0
75%           394.2     6.6         18.0
max          1772.0    16.7        258.0


In [6]:
m = base_map(zoom=14)
buildings[["bldg_id", "name", "dist_water_m", "elev_m", "dist_road_m", "geometry"]].explore(
    m=m, column="dist_water_m", cmap="Blues_r", scheme="quantiles", k=5,
    name="Distance to water (dark = close)")
roads[["highway", "name", "geometry"]].explore(m=m, color="black", style_kwds={"weight": 1}, name="Roads")
folium.LayerControl().add_to(m)
m

In [7]:
# ============================================================
# Block 4.6: smart building table + likely damage + loss
# ============================================================
# We turn raw map data into facts people can ask about:
#   floors     : estimated from height (about 3 m per floor)
#   use        : from a place (POI) at the building > map class > INFERRED
#   use_source : says whether "use" is known or inferred (be honest with users!)
#   red_flag   : simulated "likely damaged by storm surge" = near water AND low ground
#   loss_usd   : value x damage %
# Wind: every building was inside Milton's strongest-wind zone (Cat 3, see 4.2b map).

# --- Knobs: edit and re-run --------------------------------------------
WIND_DAMAGE_PCT = 15        # % loss from Cat 3 wind, for every building
NEAR_WATER_M    = 300       # "near water" = closer than this to the shore (m)
LOW_GROUND_M    = 3         # "low ground" = lower than this above sea level (m)
SURGE_EXTRA_PCT = 25        # extra % loss for near-water + low-ground buildings
COST_PER_M2 = {"apartment": 2000, "hotel": 2600, "resort": 2600, "commercial": 2200,
               "restaurant": 2200, "school": 2500, "garage": 600, "shed": 400, "roof": 400}
OTHER_COST  = 1800          # USD per m2 per floor for anything else (e.g. houses)
# --------------------------------------------------------------------------

# 1) Floors from height
buildings["floors"] = (buildings["height_m"] / 3).round().clip(lower=1)

# 2) Attach the nearest place (POI) within 15 m of each building
near_place = gpd.sjoin_nearest(
    buildings[["bldg_id", "geometry"]].to_crs(METRIC_CRS),
    places[["place_name", "place_cat", "geometry"]].to_crs(METRIC_CRS),
    max_distance=15, distance_col="d").sort_values("d").drop_duplicates("bldg_id").set_index("bldg_id")
buildings["place_cat"] = buildings["bldg_id"].map(near_place["place_cat"])
buildings["name"]      = buildings["name"].fillna(buildings["bldg_id"].map(near_place["place_name"]))

# 3) Use: known if we have it, otherwise a simple, explainable guess
def infer_use(r):
    if isinstance(r["place_cat"], str): return r["place_cat"].replace("_", " "), "known (place)"
    if isinstance(r["class"], str):     return r["class"], "known (map tag)"
    if r["height_m"] >= 12:             return "apartment (inferred)", "inferred"
    if r["area_m2"] < 40:               return "shed (inferred)", "inferred"
    return "house (inferred)", "inferred"
buildings[["use", "use_source"]] = buildings.apply(infer_use, axis=1, result_type="expand")
buildings["use"] = buildings["use"].str.lower()

# 4) Value, likely damage, loss
def cost_per_m2(use):
    for word, cost in COST_PER_M2.items():
        if word in use:
            return cost
    return OTHER_COST
buildings["value_usd"]  = buildings["area_m2"] * buildings["floors"].fillna(1) * buildings["use"].map(cost_per_m2)
buildings["red_flag"]   = (buildings["dist_water_m"] < NEAR_WATER_M) & (buildings["elev_m"] < LOW_GROUND_M)
buildings["damage_pct"] = WIND_DAMAGE_PCT + buildings["red_flag"] * SURGE_EXTRA_PCT
buildings["loss_usd"]   = (buildings["value_usd"] * buildings["damage_pct"] / 100).round(-2)

print(buildings["use_source"].value_counts(), "\n")
print(buildings["use"].value_counts().head(12), "\n")
print(f"Likely damaged (red): {buildings.red_flag.sum()} of {len(buildings)}")
print(f"Buildings 10+ floors: {(buildings.floors >= 10).sum()} "
      f"(likely damaged: {((buildings.floors >= 10) & buildings.red_flag).sum()})")
print(f"Total estimated loss: ${buildings.loss_usd.sum() / 1e6:.0f} million")

use_source
inferred           5981
known (map tag)     965
known (place)       806
Name: count, dtype: int64 

use
house (inferred)        5631
house                    749
shed (inferred)          312
private lodging          167
roof                     153
beach                     49
real estate service       48
home service              41
apartment (inferred)      38
historic site             34
hotel                     34
restaurant                29
Name: count, dtype: int64 

Likely damaged (red): 830 of 7752
Buildings 10+ floors: 10 (likely damaged: 2)
Total estimated loss: $1250 million


In [11]:
# ============================================================
# Block 4.7: ASK THE MAP (Gemini + interactive map)
# ============================================================
# Type a question in plain English -> Gemini turns it into a filter on our
# building table -> the matching buildings light up BLUE on the map, and the
# cards above the map show how many there are, their value and estimated loss.
# Gemini only translates your words; every number comes from our data.
# Map tools (like leafmap): ⛶ full screen, layer switcher (top right),
# mini-map, coordinates under the mouse, and a ruler to measure distances.
!pip -q install google-genai gradio
import json, re, time, html
import folium, gradio as gr
from folium import plugins
from google import genai

# ---------- 1. Gemini (free key saved in Colab Secrets as GEMINI_API_KEY) ----------
try:
    from google.colab import userdata
    key = userdata.get("GEMINI_API_KEY")
except Exception:
    from getpass import getpass
    key = getpass("Paste your Gemini API key: ")
gemini = genai.Client(api_key=key)
GEMINI_MODELS = ["gemini-3.8-flash", "gemini-3.5-flash", "gemini-3.1-flash-lite"]   # free Flash models

def ask_gemini(prompt):
    """Try each free Flash model; skip retired (404) or busy (503) ones."""
    for model in GEMINI_MODELS:
        for attempt in range(2):
            try:
                return gemini.models.generate_content(model=model, contents=prompt).text
            except Exception as err:
                if "404" in str(err) or "NOT_FOUND" in str(err):
                    break                              # retired name -> next model
                time.sleep(3 * (attempt + 1))          # busy -> wait, retry
    raise RuntimeError("Gemini is busy right now")

# ---------- 2. The map (keyless Esri basemaps + leafmap-style tools) ----------
ESRI = "https://server.arcgisonline.com/ArcGIS/rest/services/{}/MapServer/tile/{{z}}/{{y}}/{{x}}"
BASEMAPS = {"Light grey": "Canvas/World_Light_Gray_Base", "Streets": "World_Street_Map",
            "Satellite": "World_Imagery"}

# What you see when hovering over a building (prepared once for all buildings)
fmt_usd = lambda v: f"${v:,.0f}" if pd.notna(v) else "unknown"
HOVER = gpd.GeoDataFrame({
    "Building type": buildings["use"],
    "Height":        buildings["height_m"].map(lambda v: f"{v:.0f} m" if pd.notna(v) else "unknown"),
    "Value":         buildings["value_usd"].map(fmt_usd),
    "Loss":          buildings["loss_usd"].map(fmt_usd),
}, geometry=buildings.geometry, crs=buildings.crs)

def make_map(highlight=None):
    m = folium.Map(location=center, zoom_start=12, tiles=None, control_scale=True)
    for name, path in BASEMAPS.items():
        folium.TileLayer(ESRI.format(path), attr="Esri", name=name).add_to(m)
    # Hurricane Milton: hurricane-force wind footprint (off by default) + the track
    swaths[64].explore(m=m, color="#f2a900", style_kwds={"fillOpacity": 0.12, "weight": 1},
                       tooltip=False, name="Hurricane-force winds (64 kt+)", show=False)
    track_line.explore(m=m, color="#6a3ea1", style_kwds={"weight": 4}, tooltip=False,
                       name="Hurricane Milton track")
    # Buildings
    HOVER.explore(m=m, color="#9aa0a6", style_kwds={"fillOpacity": 0.15, "weight": 0.4},
                  name="All buildings")
    HOVER[buildings["red_flag"]].explore(m=m, color="#e0452c", style_kwds={"fillOpacity": 0, "weight": 1.5},
                                         name="Likely damaged (simulated)")
    if highlight is not None and len(highlight):
        HOVER.loc[highlight.index].explore(m=m, color="#1f6feb", style_kwds={"fillOpacity": 0.6, "weight": 2},
                                           name="Your question")
        x0, y0, x1, y1 = highlight.total_bounds
        m.fit_bounds([[y0, x0], [y1, x1]], max_zoom=17)
    plugins.Fullscreen(position="topleft").add_to(m)
    plugins.MiniMap(toggle_display=True, position="bottomright").add_to(m)
    plugins.MousePosition(position="bottomleft", prefix="Lat/Lon:").add_to(m)
    plugins.MeasureControl(position="topleft", primary_length_unit="meters").add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    return m._repr_html_()

# ---------- 3. Question -> filter -> highlighted buildings ----------
COLUMNS = """use          : what the building is used for (lowercase). Common values: {uses}.
               Inferred ones end with '(inferred)'. Match with use.str.contains('word').
use_source   : 'known (place)', 'known (map tag)' or 'inferred' (guessed from size/height)
name         : building or business name (often empty)
floors       : number of floors (estimated from height, often empty)
height_m     : building height (m)
area_m2      : footprint area (m2)
value_usd    : estimated building value (USD)
dist_water_m : distance to the shoreline (m)
elev_m       : ground height above sea level (m)
dist_road_m  : distance to the nearest road (m)
nearest_road : trunk, primary, secondary, tertiary, residential, unclassified, service
red_flag     : True if likely damaged (simulated: near water and low ground)
loss_usd     : estimated loss (USD)"""

EXAMPLES = """Q: Apartments taller than 10 floors that are likely damaged
A: {"query": "use.str.contains('apartment') and floors > 10 and red_flag == True", "title": "Damaged apartment towers over 10 floors"}
Q: Confirmed houses only (not guessed) within 100 m of the water
A: {"query": "use.str.contains('house') and use_source != 'inferred' and dist_water_m < 100", "title": "Confirmed houses near the water"}
Q: The 20 buildings with the highest loss
A: {"query": "", "title": "Top 20 losses", "sort_by": "loss_usd", "top": 20}
Q: (previous filter: use.str.contains('apartment')) Only those within 100 m of the water
A: {"query": "use.str.contains('apartment') and dist_water_m < 100", "title": "Apartments within 100 m of the water"}"""

def question_to_filter(question, last_q):
    uses = ", ".join(buildings["use"].value_counts().head(15).index)
    prompt = ("You turn questions about buildings into a pandas DataFrame.query() filter.\n"
              "Columns:\n" + COLUMNS.format(uses=uses) + "\n\nExamples:\n" + EXAMPLES +
              f"\n\nPrevious filter: {last_q or 'none'}. If the new question refines it "
              "('of those', 'only', 'which of them'), combine both with ' and '; otherwise ignore it.\n"
              "Reply with ONLY one JSON object with keys 'query' and 'title' (optionally 'sort_by' "
              "and 'top'). Use only the columns above.\nQ: " + question + "\nA:")
    found = re.search(r"\{.*\}", ask_gemini(prompt), re.S)
    if not found:
        raise ValueError("no filter returned")
    return json.loads(found.group(0))

def run_filter(spec):
    q = (spec.get("query") or "").strip()
    if any(bad in q for bad in ["__", "import", "lambda", "@", "eval", "exec", "open("]):
        raise ValueError("unsafe filter")
    res = buildings.query(q, engine="python") if q else buildings
    if spec.get("sort_by") in buildings.columns:
        res = res.sort_values(spec["sort_by"], ascending=False)
    if spec.get("top"):
        res = res.head(int(spec["top"]))
    return q, res

def kpi_html(res, title="Whole study area"):
    def card(label, value, colour):
        return (f"<div style='flex:1;min-width:130px;padding:10px 14px;border-radius:12px;"
                f"background:{colour}14;border:1px solid {colour}55'>"
                f"<div style='font-size:12px;color:#555'>{label}</div>"
                f"<div style='font-size:22px;font-weight:700;color:{colour}'>{value}</div></div>")
    return (f"<div style='font-size:14px;margin:2px 0 6px'><b>{html.escape(title)}</b></div>"
            "<div style='display:flex;gap:10px;flex-wrap:wrap'>"
            + card("Buildings", f"{len(res):,}", "#1f6feb")
            + card("Likely damaged", f"{int(res['red_flag'].sum()):,}", "#d1442e")
            + card("Exposed value", f"${res['value_usd'].sum() / 1e6:,.0f}M", "#6f42c1")
            + card("Estimated loss", f"${res['loss_usd'].sum() / 1e6:,.1f}M", "#b35900")
            + "</div>")

def results_table(res):
    t = HOVER.loc[res.sort_values("loss_usd", ascending=False).head(200).index]
    return pd.DataFrame(t.drop(columns="geometry"))

def highlight(question, last_q):
    try:
        spec = question_to_filter(question, last_q)
        q, res = run_filter(spec)
    except Exception:
        msg = "Couldn't understand that (or Gemini is busy) – try rephrasing"
        return gr.update(), kpi_html(buildings.iloc[0:0], msg), gr.update(), last_q
    return make_map(res), kpi_html(res, spec.get("title", "Your question")), results_table(res), q

def reset():
    return make_map(), kpi_html(buildings), results_table(buildings), "", ""

# ---------- 4. The app ----------
with gr.Blocks(title="Ask the map: Hurricane Milton") as app:
    gr.Markdown("## 🌀 Ask the map: Hurricane Milton, Siesta Key (October 2024)")
    kpis = gr.HTML(kpi_html(buildings))
    last_q = gr.State("")
    with gr.Row(equal_height=False):
        with gr.Column(scale=3):
            the_map = gr.HTML(make_map())
        with gr.Column(scale=1, min_width=280):
            box = gr.Textbox(label="Ask the map", lines=2,
                             placeholder="e.g. confirmed houses within 100 m of the water")
            go = gr.Button("Highlight", variant="primary")
            with gr.Accordion("💡 Example questions", open=True):
                gr.Examples(["Apartments taller than 10 floors that are likely damaged",
                             "Only those within 100 m of the water",
                             "Confirmed houses only (not guessed)",
                             "Restaurants and shops on ground lower than 2 m",
                             "Buildings more than 200 m from any road",
                             "The 20 buildings with the highest loss"], inputs=box)
            reset_btn = gr.Button("↺ Reset map")
    with gr.Accordion("📋 Highlighted buildings (top 200 by loss)", open=False):
        table = gr.Dataframe(results_table(buildings), interactive=False, wrap=True)
    with gr.Accordion("ℹ️ How this works", open=False):
        gr.Markdown("1. Gemini turns your question into a filter on the building table.\n"
                    "2. Python finds the matching buildings and adds up their value and loss.\n"
                    "3. The map highlights them in blue. Red outlines = *simulated* likely damage "
                    "(near water + low ground). Building types ending in *(inferred)* are guesses from "
                    "size and height; the others come from map data.\n\n"
                    "Data: Overture Maps, OpenStreetMap, Copernicus DEM, NOAA IBTrACS.")
    for trigger in (go.click, box.submit):
        trigger(highlight, [box, last_q], [the_map, kpis, table, last_q])
    reset_btn.click(reset, None, [the_map, kpis, table, box, last_q])

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3a56e1091b545b3c71.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
